# 配送时长中的 MSE 与 MAE

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

使用同一组本地真实数据预测，逐行比较平方误差与绝对误差。

In [2]:
def left_fold_mean(values):
    values = np.asarray(values, dtype=np.float64)
    total = 0.0
    for value in values:
        total += float(value) / values.size
    return total


def regression_losses(targets, predictions):
    targets = np.asarray(targets, dtype=np.float64)
    predictions = np.asarray(predictions, dtype=np.float64)
    if targets.shape != predictions.shape or targets.ndim != 1 or targets.size == 0:
        raise ValueError("targets and predictions must be equal non-empty vectors")
    if not np.isfinite(targets).all() or not np.isfinite(predictions).all():
        raise ValueError("targets and predictions must be finite")
    residuals = predictions - targets
    squared = residuals ** 2
    absolute = np.abs(residuals)
    return {
        "residuals": residuals,
        "squared": squared,
        "absolute": absolute,
        "mse": left_fold_mean(squared),
        "mae": left_fold_mean(absolute),
        "mse_per_element_gradients": 2.0 * residuals,
        "mae_per_element_subgradients": np.sign(residuals),
        "mae_differentiable": residuals != 0.0,
        "mse_mean_gradients": 2.0 * residuals / targets.size,
        "mae_mean_subgradients": np.sign(residuals) / targets.size,
    }

In [3]:
import struct
import zlib


def write_strict_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = json.dumps(
        value,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
        allow_nan=False,
    ) + "\n"
    path.write_text(payload, encoding="utf-8", newline="\n")


def _png_chunk(kind, payload):
    return (
        struct.pack(">I", len(payload))
        + kind
        + payload
        + struct.pack(">I", zlib.crc32(kind + payload) & 0xFFFFFFFF)
    )


def write_pattern_plot(path, *, title, description, non_color_encoding, series):
    width, height = 960, 540
    pixels = bytearray([248, 250, 252] * width * height)

    def set_pixel(x, y, color):
        if 0 <= x < width and 0 <= y < height:
            offset = (y * width + x) * 3
            pixels[offset : offset + 3] = bytes(color)

    def line(x0, y0, x1, y1, color, dashed=False):
        dx = abs(x1 - x0)
        sx = 1 if x0 < x1 else -1
        dy = -abs(y1 - y0)
        sy = 1 if y0 < y1 else -1
        error = dx + dy
        step = 0
        while True:
            if not dashed or (step // 7) % 2 == 0:
                set_pixel(x0, y0, color)
                set_pixel(x0, y0 + 1, color)
            if x0 == x1 and y0 == y1:
                break
            doubled = 2 * error
            if doubled >= dy:
                error += dy
                x0 += sx
            if doubled <= dx:
                error += dx
                y0 += sy
            step += 1

    left, right, top, bottom = 80, 900, 55, 470
    line(left, bottom, right, bottom, (30, 41, 59))
    line(left, top, left, bottom, (30, 41, 59))
    flattened = [abs(float(value)) for _, values in series for value in values]
    maximum = max(flattened, default=1.0) or 1.0
    colors = ((35, 85, 155), (178, 72, 56), (52, 125, 83), (111, 78, 153))
    for series_index, (_, values) in enumerate(series):
        values = [float(value) for value in values]
        if not values:
            continue
        for index, value in enumerate(values):
            x = left + round(index * (right - left) / max(1, len(values) - 1))
            y = bottom - round(abs(value) / maximum * (bottom - top - 20))
            if index:
                previous = float(values[index - 1])
                previous_x = left + round((index - 1) * (right - left) / max(1, len(values) - 1))
                previous_y = bottom - round(abs(previous) / maximum * (bottom - top - 20))
                line(
                    previous_x,
                    previous_y,
                    x,
                    y,
                    colors[series_index % len(colors)],
                    dashed=series_index % 2 == 1,
                )
            marker = 4 + series_index
            for marker_y in range(y - marker, y + marker + 1):
                for marker_x in range(x - marker, x + marker + 1):
                    if series_index % 2 == 0:
                        draw = abs(marker_x - x) + abs(marker_y - y) <= marker
                    else:
                        draw = (
                            abs(marker_x - x) == marker
                            or abs(marker_y - y) == marker
                        )
                    if draw:
                        set_pixel(
                            marker_x,
                            marker_y,
                            colors[series_index % len(colors)],
                        )

    raw = b"".join(
        b"\x00" + bytes(pixels[row * width * 3 : (row + 1) * width * 3])
        for row in range(height)
    )
    metadata = {
        "Title": title,
        "Description": description,
        "NonColorEncoding": non_color_encoding,
        "Software": "ML Atlas Phase 26 deterministic stdlib PNG writer",
    }
    png = bytearray(b"\x89PNG\r\n\x1a\n")
    png.extend(
        _png_chunk(
            b"IHDR",
            struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0),
        )
    )
    for key, value in metadata.items():
        png.extend(_png_chunk(b"tEXt", key.encode("latin1") + b"\x00" + value.encode("latin1")))
    png.extend(_png_chunk(b"IDAT", zlib.compress(raw, level=9)))
    png.extend(_png_chunk(b"IEND", b""))
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(bytes(png))

In [4]:
DATASET_PATH = Path("../../datasets/loss-functions/lade-delivery-jilin.csv")
DATASET_MANIFEST_PATH = Path("../../datasets/loss-functions/lade-delivery-jilin-manifest.json")
SUMMARY_PATH = Path("outputs/regression-loss-summary.json")
PLOT_PATH = Path("outputs/delivery-losses.png")

In [5]:
frame = pd.read_csv(DATASET_PATH)
dataset_manifest = json.loads(DATASET_MANIFEST_PATH.read_text(encoding="utf-8"))
targets = frame["delivery_duration_minutes"].to_numpy(dtype=np.float64)
predictions = np.full(targets.shape, 175.0, dtype=np.float64)
evaluated = regression_losses(targets, predictions)

rows = []
for index, course_row_id in enumerate(frame["course_row_id"].astype(str)):
    rows.append({
        "courseRowId": course_row_id,
        "targetMinutes": float(targets[index]),
        "predictionMinutes": float(predictions[index]),
        "residualMinutes": float(evaluated["residuals"][index]),
        "mseLoss": float(evaluated["squared"][index]),
        "maeLoss": float(evaluated["absolute"][index]),
        "msePerElementGradient": float(evaluated["mse_per_element_gradients"][index]),
        "maePerElementSubgradient": float(evaluated["mae_per_element_subgradients"][index]),
        "maeDifferentiable": bool(evaluated["mae_differentiable"][index]),
        "mseMeanObjectiveGradient": float(evaluated["mse_mean_gradients"][index]),
        "maeMeanObjectiveSubgradient": float(evaluated["mae_mean_subgradients"][index]),
    })

rows_by_id = {row["courseRowId"]: row for row in rows}
representative_rows = [
    {"role": reference["role"], **rows_by_id[reference["courseRowId"]]}
    for reference in dataset_manifest["teachingReference"]["representativeRows"]
]
high_contribution_rows = sorted(
    rows,
    key=lambda row: (-row["mseLoss"], row["courseRowId"]),
)[:5]
histogram_counts, histogram_edges = np.histogram(
    evaluated["absolute"],
    bins=np.asarray([0, 15, 30, 60, 120, 240, 480, 960, 1920, 3840], dtype=np.float64),
)
summary = {
    "contractVersion": "loss-functions-phase-26-output-v1",
    "topicId": "delivery-losses",
    "dataset": {
        "datasetId": dataset_manifest["datasetId"],
        "sha256": dataset_manifest["published"]["sha256"],
        "rowCount": int(targets.size),
    },
    "referencePredictionMinutes": 175,
    "aggregate": {
        "mse": evaluated["mse"],
        "mae": evaluated["mae"],
        "rowCount": int(targets.size),
    },
    "rows": rows,
    "representativeRows": representative_rows,
    "highContributionRows": high_contribution_rows,
    "distribution": {
        "metric": "absolute residual minutes",
        "binEdges": [float(value) for value in histogram_edges],
        "counts": [int(value) for value in histogram_counts],
    },
    "plot": {
        "path": "outputs/delivery-losses.png",
        "width": 960,
        "height": 540,
        "nonColorEncoding": "MSE solid diamond line; MAE dashed square line",
    },
}
write_strict_json(SUMMARY_PATH, summary)
write_pattern_plot(
    PLOT_PATH,
    title="Delivery loss distribution",
    description="Real LaDe delivery rows summarized by MSE and MAE contribution scale.",
    non_color_encoding="MSE solid diamond line; MAE dashed square line",
    series=(
        ("MSE solid diamond", [row["mseLoss"] for row in high_contribution_rows]),
        ("MAE dashed square", [row["maeLoss"] for row in high_contribution_rows]),
    ),
)
print(json.dumps({
    "topicId": summary["topicId"],
    "rowCount": summary["aggregate"]["rowCount"],
    "mse": summary["aggregate"]["mse"],
    "mae": summary["aggregate"]["mae"],
}, sort_keys=True, allow_nan=False))

{"mae": 106.08492758236584, "mse": 21178.123380550926, "rowCount": 31415, "topicId": "delivery-losses"}
